In [1]:
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

In [2]:
from model import SeqNN

In [3]:
import schedulefree

In [4]:
class HiCDataset(Dataset):
    def __init__(self, data_files):
        self.data = []  # To store all sequences and HiC vectors
        
        # Load and process the data files
        for file in data_files:
            file_data = torch.load(file, weights_only=True)
            
            for data in file_data:
                ohe_sequence, hic_vector = data

                # Process the OHE sequence
                ohe_sequence = ohe_sequence.squeeze(0)  # Remove singleton dimension
                
                # Ensure the sequence has the correct shape
                assert ohe_sequence.shape[0] == 4, f"Expected 4 channels, but got {ohe_sequence.shape[0]}"
                assert len(ohe_sequence.shape) == 2, f"Expected 2D shape for sequence, but got {ohe_sequence.shape}"
                
                # Add processed pair to the data list
                self.data.append((ohe_sequence, hic_vector))
        
        print(f"Total sequences loaded: {len(self.data)}")
        
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        # Fetch the preprocessed (ohe_sequence, hic_vector) pair from memory
        ohe_sequence, hic_vector = self.data[idx]
        return ohe_sequence, hic_vector

In [5]:
train_files = ["/scratch1/smaruj/train_pytorch_akita/mouse/fold2_0.pt"]

In [6]:
train_dataset = HiCDataset(train_files)

Total sequences loaded: 100


In [7]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=False, num_workers=4, pin_memory=True)

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
model = SeqNN().to(device)

In [10]:
from torchinfo import summary

summary(model, input_size=(2, 4, 1310720), col_names=["output_size", "num_params"])

Layer (type:depth-idx)                   Output Shape              Param #
SeqNN                                    [2, 1, 130305]            --
├─StochasticReverseComplement: 1-1       [2, 4, 1310720]           --
├─StochasticShift: 1-2                   [2, 4, 1310720]           --
├─ConvBlock: 1-3                         [2, 96, 655360]           --
│    └─Conv1d: 2-1                       [2, 96, 1310720]          4,224
│    └─BatchNorm1d: 2-2                  [2, 96, 1310720]          192
│    └─MaxPool1d: 2-3                    [2, 96, 655360]           --
├─ConvTower: 1-4                         [2, 96, 640]              --
│    └─Sequential: 2-4                   [2, 96, 640]              --
│    │    └─ReLU: 3-1                    [2, 96, 655360]           --
│    │    └─Conv1d: 3-2                  [2, 96, 655360]           46,080
│    │    └─BatchNorm1d: 3-3             [2, 96, 655360]           192
│    │    └─MaxPool1d: 3-4               [2, 96, 327680]           --
│    │

In [11]:
model.train()

SeqNN(
  (stochastic_reverse_complement): StochasticReverseComplement()
  (stochastic_shift): StochasticShift()
  (conv_block_1): ConvBlock(
    (conv): Conv1d(4, 96, kernel_size=(11,), stride=(1,), padding=(5,), bias=False)
    (batch_norm): BatchNorm1d(96, eps=0.001, momentum=0.0735, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_tower): ConvTower(
    (conv_tower): Sequential(
      (0): ReLU()
      (1): Conv1d(96, 96, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (2): BatchNorm1d(96, eps=0.001, momentum=0.0735, affine=True, track_running_stats=True)
      (3): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): ReLU()
      (5): Conv1d(96, 96, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
      (6): BatchNorm1d(96, eps=0.001, momentum=0.0735, affine=True, track_running_stats=True)
      (7): MaxPool1d(kernel_size=2, stride=2, padd

In [12]:
example = next(iter(train_loader))

In [13]:
input, target = example

In [14]:
input.shape, target.shape

(torch.Size([2, 4, 1310720]), torch.Size([2, 1, 130305]))

In [15]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def plot_map(matrix, vmin=-0.6, vmax=0.6, palette="RdBu_r", width=5, height=5):
    fig, axes = plt.subplots(1, 1, figsize=(width, height))

    sns.heatmap(
        matrix,
        vmin=vmin,
        vmax=vmax,
        cbar=False,
        cmap=palette,
        square=True,
        xticklabels=False,
        yticklabels=False,
        ax=axes
    )

    plt.tight_layout()
    plt.show()
    

# Helper function to set diagonal elements to a specific value
def set_diag(matrix, value, k):
    # Explicitly set the diagonal to 'value' (in this case, np.nan) for each k
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value


def from_upper_triu(vector_repr, matrix_len, num_diags):
    # Ensure vector_repr is a NumPy array (if it's a PyTorch tensor, convert it)
    if isinstance(vector_repr, torch.Tensor):
        vector_repr = vector_repr.detach().flatten().cpu().numpy()  # Flatten and convert to NumPy array

    # Initialize a zero matrix of shape (matrix_len, matrix_len)
    z = np.zeros((matrix_len, matrix_len))

    # Get the indices for the upper triangular matrix
    triu_tup = np.triu_indices(matrix_len, num_diags)

    # Assign the values from the vector_repr to the upper triangular part of the matrix
    z[triu_tup] = vector_repr

    # Set the diagonals specified by num_diags to np.nan
    for i in range(-num_diags + 1, num_diags):
        set_diag(z, np.nan, i)

    # Ensure the matrix is symmetric
    return z + z.T

In [ ]:
matrix = from_upper_triu(target[0,:,:], matrix_len=512, num_diags=2)
plot_map(matrix)

In [ ]:
input.shape

In [16]:
# forward-like pass

x, reverse_bool = model.stochastic_reverse_complement(input)
x = x.to(device)

print(reverse_bool)

x = model.stochastic_shift(x)
x = x.to(device)

x = model.conv_block_1(x)

x = x.to(device)
x = model.conv_tower(x)

x = model.residual1d_block1(x) 
x = model.residual1d_block2(x) 
x = model.residual1d_block3(x)
x = model.residual1d_block4(x)
x = model.residual1d_block5(x)
x = model.residual1d_block6(x)
x = model.residual1d_block7(x)
x = model.residual1d_block8(x)

x = model.conv_reduce(x)

x = model.one_to_two(x)

x = model.concat_dist(x)

print(x.shape)

x = model.conv2d_block(x)

x = model.symmetrize_2d(x)

print(x.shape)

x = model.residual2d_block1(x)
x = model.residual2d_block2(x)
x = model.residual2d_block3(x)
x = model.residual2d_block4(x)
x = model.residual2d_block5(x)
x = model.residual2d_block6(x)
x = model.cropping_2d(x)

print(x.shape)

x = model.upper_tri(x, reverse_bool)

print(x.shape)

x = model.final(x)

print(x.shape)


tensor([False, False])
torch.Size([2, 65, 640, 640])
torch.Size([2, 48, 640, 640])
torch.Size([2, 48, 512, 512])
torch.Size([2, 48, 130305])
torch.Size([2, 1, 130305])


In [ ]:
import torch.nn as nn

class UpperTri(nn.Module):
    ''' Unroll matrix to its upper triangular portion. '''
    def __init__(self, diagonal_offset=2):
        super(UpperTri, self).__init__()
        self.diagonal_offset = diagonal_offset

    def forward(self, inputs, reverse_complement_flags):
        # Get the batch size, features_dim, and mat_size from the shape of the input tensor
        batch_size, features_dim, mat_size, _ = inputs.shape
        
        # Generate the upper triangular indices with the specified diagonal offset
        triu_tup = torch.triu_indices(mat_size, mat_size, self.diagonal_offset)

        # Flatten the input tensor to shape [batch_size, features_dim, mat_size^2]
        unroll_repr = inputs.reshape(batch_size, features_dim, mat_size * mat_size)
        
        # Convert triu indices to flattened indices
        triu_index = triu_tup[0] * mat_size + triu_tup[1]  # Row * mat_size + col
        triu_index = triu_index.to(unroll_repr.device)
        
        # Unsqueeze triu_index to make it [mat_size^2] and expand it across all the batch examples and channels
        triu_index = triu_index.unsqueeze(0).unsqueeze(0).expand(batch_size, features_dim, -1)

        # Generate the flipped version of the input by rotating along the main diagonal
        flipped_repr = unroll_repr.flip(2)  # Flip along the flattened matrix rows

        unroll_repr = unroll_repr.to(device)
        flipped_repr = flipped_repr.to(device)  
        reverse_complement_flags = reverse_complement_flags.to(device)
        
        # Use reverse_complement_flags to select between original and flipped maps
        upper_tri = torch.where(reverse_complement_flags.view(batch_size, 1, 1).expand(batch_size, features_dim, -1),
                                torch.gather(unroll_repr, 2, triu_index),
                                torch.gather(flipped_repr, 2, triu_index))

        return upper_tri

In [ ]:
upper_tri_class = UpperTri()

In [ ]:
reverse_bool

In [ ]:
check = upper_tri_class(x, reverse_bool)

In [ ]:
check.shape

In [ ]:
original = x[1, 27, :,:]

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
original_np = original.cpu().detach().numpy()

# Plot the matrix using matshow
plt.matshow(original_np, cmap='viridis')
plt.colorbar()
plt.title("Heatmap of the Tensor")
plt.show()

In [ ]:
vector = check[1, 27, :]

In [ ]:
trans = from_upper_triu(vector, matrix_len=512, num_diags=2)

In [ ]:
plt.matshow(trans, cmap='viridis')
plt.colorbar()
plt.title("Heatmap of the Tensor")
plt.show()

In [ ]:
plot_map(from_upper_triu(x[0,:,:], matrix_len=512, num_diags=2))